# Aula 2, Tool calling

Notebook executável que acompanha a aula [02-tool-calling.md](../../lessons/modulo-10-agentes/02-tool-calling.md).

## O que vamos fazer aqui

Agora o LLM decide a ferramenta, pedindo a chamada em JSON. Vamos montar um registro de
ferramentas e um despachante que valida e executa o pedido. Testamos com pedidos simulados,
inclusive um inválido. Python puro.

## Registro de ferramentas e despachante

Cada ferramenta tem nome, descrição e função. O despachante lê o JSON do modelo, valida (JSON
válido, ferramenta existente, argumento presente) e executa, tratando erros.

In [ ]:
import re
import json
import ast
import operator

OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
       ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}


def calcular(expressao):
    def ev(no):
        if isinstance(no, ast.Constant) and isinstance(no.value, (int, float)):
            return no.value
        if isinstance(no, ast.BinOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.left), ev(no.right))
        if isinstance(no, ast.UnaryOp) and type(no.op) in OPS:
            return OPS[type(no.op)](ev(no.operand))
        raise ValueError("expressão não permitida")
    return ev(ast.parse(str(expressao), mode="eval").body)


def buscar(consulta):
    base = {"derivada": "A derivada mede a taxa de variação de uma função."}
    for chave, texto in base.items():
        if chave in str(consulta).lower():
            return texto
    return "Não encontrei no material."


FERRAMENTAS = {
    "calcular": ("Resolve uma expressão aritmética.", calcular, "expressao"),
    "buscar": ("Busca um tema na base de notas de aula.", buscar, "consulta"),
}


def despachar(pedido_json):
    m = re.search(r"\{.*\}", pedido_json, re.DOTALL)
    if not m:
        return "Erro: não veio um JSON."
    try:
        pedido = json.loads(m.group(0))
    except json.JSONDecodeError:
        return "Erro: JSON inválido."
    nome = pedido.get("ferramenta")
    if nome not in FERRAMENTAS:
        return f"Erro: ferramenta '{nome}' não existe."
    _, funcao, arg_nome = FERRAMENTAS[nome]
    if arg_nome not in pedido:
        return f"Erro: falta o argumento '{arg_nome}'."
    try:
        return funcao(pedido[arg_nome])
    except Exception as erro:
        return f"Erro ao executar: {erro}"


pedidos = [
    '{"ferramenta": "calcular", "expressao": "28*3/4"}',
    '{"ferramenta": "buscar", "consulta": "o que é a derivada?"}',
    '{"ferramenta": "voar", "destino": "lua"}',
]
for p in pedidos:
    print(despachar(p))

O despachante executa os dois primeiros pedidos (21.0 e o trecho da derivada) e trata o
terceiro com erro claro, pois a ferramenta "voar" não existe. O modelo decide e pede em JSON, o
nosso código valida e executa. Reusamos a extração e validação de JSON do Módulo 8.

## Caminho opcional, o LLM decidindo via Ollama

In [ ]:
def descricao_ferramentas():
    return "\n".join(f"- {nome}: {desc} (argumento: {arg})"
                     for nome, (desc, _, arg) in FERRAMENTAS.items())


try:
    import ollama

    pergunta = "quanto é 15 * 12?"
    prompt = (
        "Você tem estas ferramentas:\n" + descricao_ferramentas() + "\n\n"
        "Responda APENAS com um JSON {\"ferramenta\": ..., <argumento>: ...} "
        f"para resolver: {pergunta}"
    )
    r = ollama.chat(model="llama3.1", messages=[{"role": "user", "content": prompt}])
    print("Pedido do modelo:", r["message"]["content"])
    print("Resultado:", despachar(r["message"]["content"]))
except Exception as erro:
    print("Ollama não disponível; os pedidos simulados acima já demonstram o despachante.")
    print("Detalhe:", erro)

Com o Ollama, o próprio modelo escolhe a ferramenta e os argumentos, e o despachante
executa. Para o projeto, monte um agente com tool calling pelo LLM e tratamento de erros.